# Hiraeth — QLoRA Fine-tune on Kaggle (2x T4 / P100)

Workflow: code lives on **GitHub**, data is uploaded as a **Kaggle Dataset**, this notebook clones the repo, prepares data, smoke-tests, trains, merges, and zips the result for download.

**Read `docs/TRAINING_GUIDE.md` in the repo for the full walkthrough and troubleshooting** — this notebook follows that guide step by step.

**Before running:**
1. Settings > Accelerator > **GPU T4 x2** (or P100 x2), Internet > **On**.
2. Attach your Hiraeth Atlas Kaggle Dataset via **Add Input**.
3. Update `DATASET_NAME` / `RAW_FILENAME` in Step 4 to match your attached dataset.

In [ ]:
!nvidia-smi

## 1. Clone the Hiraeth repo

In [ ]:
REPO_NAME = "Hiraeth"  # matches github.com/jadhavdurvesh/Hiraeth exactly (case-sensitive)

!git clone https://github.com/jadhavdurvesh/{REPO_NAME}.git
!ls {REPO_NAME}/scripts/

## 2. Install dependencies — with `--no-deps`

Kaggle notebooks ship with a PyTorch build matched to their CUDA driver. A plain `pip install -r requirements.txt` can silently upgrade torch and break GPU support. `--no-deps` avoids that — see `docs/TRAINING_GUIDE.md` for why.

In [ ]:
!pip install -q --no-deps -r {REPO_NAME}/scripts/requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
# If CUDA available is False here, STOP and see the Troubleshooting section
# in docs/TRAINING_GUIDE.md before continuing.

# Optional: install Flash Attention 2 for faster/safer packing on T4
# (skip on P100 - not supported there; train.py falls back safely either way)
# !pip install -q flash-attn --no-build-isolation

## 3. Check your attached dataset

In [ ]:
!ls /kaggle/input/

## 4. Prepare the dataset

If you uploaded the raw Hiraeth Atlas merged file, this converts it to chat-formatted train/val JSONL. If you already uploaded pre-formatted `train.jsonl`/`val.jsonl`, skip this cell and point `TRAIN_FILE`/`VAL_FILE` in Step 5 directly at `/kaggle/input/<your-dataset-name>/train.jsonl` etc.

In [ ]:
DATASET_NAME = "hiraeth-atlas"        # <-- change to your Kaggle Dataset's folder name under /kaggle/input/
RAW_FILENAME = "dmj_dataset_v1.0.0.jsonl"  # <-- change to your actual filename

!mkdir -p /kaggle/working/data
!python {REPO_NAME}/scripts/prepare_dataset.py \
    --input /kaggle/input/{DATASET_NAME}/{RAW_FILENAME} \
    --output_dir /kaggle/working/data \
    --val_split 0.02 \
    --system_prompt "You are Hiraeth, a helpful, precise AI assistant."

In [ ]:
# If you skipped Step 4 because you uploaded pre-formatted files, change these instead:
TRAIN_FILE = "/kaggle/working/data/train.jsonl"
VAL_FILE = "/kaggle/working/data/val.jsonl"

## 5. Smoke test (strongly recommended before the full run)

20 steps, a few minutes. Confirms GPU detection, correct fp16/bf16 selection, and multi-GPU handling work before committing hours of GPU quota. Check the printed logs against `docs/TRAINING_GUIDE.md` Step 5.

In [ ]:
!torchrun --standalone --nproc_per_node=2 {REPO_NAME}/scripts/train.py \
    --train_file {TRAIN_FILE} \
    --val_file {VAL_FILE} \
    --output_dir /kaggle/working/hiraeth-smoketest \
    --max_steps 20

## 6. Full training run (QLoRA, sharded across both GPUs)

In [ ]:
!torchrun --standalone --nproc_per_node=2 {REPO_NAME}/scripts/train.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --train_file {TRAIN_FILE} \
    --val_file {VAL_FILE} \
    --output_dir /kaggle/working/hiraeth-qlora \
    --num_train_epochs 3 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --learning_rate 2e-4 \
    --max_seq_length 2048

## 7. Merge adapter into a standalone model

In [ ]:
!python {REPO_NAME}/scripts/merge_and_save.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir /kaggle/working/hiraeth-qlora \
    --output_dir /kaggle/working/hiraeth-merged

## 8. Eval spot-check (optional but recommended)

In [ ]:
!python {REPO_NAME}/scripts/run_eval.py \
    --model_dir /kaggle/working/hiraeth-merged \
    --prompts_file {REPO_NAME}/eval/eval_prompts.json \
    --output_dir /kaggle/working/eval_reports

## 9. Zip the trained model for download

`/kaggle/working` is wiped when the session ends — grab this before you close the notebook.

In [ ]:
!zip -r -q /kaggle/working/hiraeth-merged.zip /kaggle/working/hiraeth-merged
!ls -lh /kaggle/working/hiraeth-merged.zip

### Alternative: push straight to Hugging Face Hub instead of downloading a zip

A 7B model zip is ~14-15GB — often easier to push to HF Hub. Add an HF token as a Kaggle Secret (`Add-ons > Secrets`) named `HF_TOKEN`, then:

In [ ]:
# from kaggle_secrets import UserSecretsClient
# hf_token = UserSecretsClient().get_secret("HF_TOKEN")
# !huggingface-cli login --token {hf_token}
# !python {REPO_NAME}/scripts/merge_and_save.py \
#     --base_model Qwen/Qwen2.5-7B-Instruct \
#     --adapter_dir /kaggle/working/hiraeth-qlora \
#     --output_dir /kaggle/working/hiraeth-merged \
#     --push_to_hub_id YOUR_HF_USERNAME/hiraeth-7b